# 06.03 — Async Query Runner

This notebook demonstrates the **async execution path**.
It runs the same filmography domain used in `06.01_fastapi_integration.ipynb`
against **in-notebook fake sessions** (no live database required) so all cells
execute cleanly in CI.

1. **Caller-owned transactions** — `CypherExecutor` (sync) and `AsyncCypherExecutor`
   (async) **never** call `commit()` or `rollback()`. The caller owns the boundary.
   Matches the MP-backend `transaction_context` that commits Postgres and Neo4j together.

2. **Parallel `Executor` / `AsyncExecutor` hierarchies** — `async def` is consumed
   only with `await`; a sync caller cannot `await`. Hence two ABCs, two executors,
   two port classes. The query-definition layer (`build` / `materialize` /
   `interpret_result`) is colour-neutral and shared unchanged.

3. **Scope = query runner only** — async inspection is deferred until a concrete use case.

**Sections:**
1. Dependency check
2. Domain model and queries
3. Sync path recap — `CypherExecutor` + `FakeSession` (caller-owned)
4. Async fake session — in-notebook `AsyncFakeSession`
5. Async read — `AsyncCypherExecutor.read()` materialises records
6. Async write — `AsyncCypherExecutor.write()` returns interpreted result
7. Caller-owned transaction demo — executor runs but does NOT commit
8. Async FastAPI route — `async def` handler + `httpx.AsyncClient`
9. `AsyncReadPort` / `AsyncQueryBackedReadPort` — DI pattern for async

## 1. Dependency check

This notebook requires `fastapi` and `httpx2`. They are **not** dependencies of
`orthograph` itself. The cell below skips the rest of the notebook with a clear
message if either package is missing.

In [ ]:
import importlib
from typing import Any, Optional

import fastapi
import httpx2 as httpx
from pydantic import BaseModel

from orthograph.cypher.query_execution import AsyncCypherExecutor, CypherExecutor
from orthograph.queries import TypedCypherReadQueryModel, TypedCypherWriteQueryModel
from orthograph.query.base_models import (
    AsyncQueryBackedReadPort,
    AsyncReadPort,
    QueryBackedReadPort,
)


_MISSING = [
    pkg for pkg in ("fastapi", "httpx2") if importlib.util.find_spec(pkg) is None
]


if _MISSING:
    msg = (
        f"Missing packages: {', '.join(_MISSING)}\n"
        "Install them with:\n"
        f"    pip install {' '.join(_MISSING)}\n"
        "This notebook is skipped — orthograph itself does not require FastAPI."
    )
    print(msg)
    raise SystemExit(msg)


print(f"fastapi {fastapi.__version__}  |  httpx2 {httpx.__version__}  — OK")

## 2. Domain model and queries

Same filmography domain as `06.01`. One `NodeModel`, one read query, one write
query. The query-definition layer is **colour-neutral** — the same query objects
drive both the sync and async executors.

In [ ]:
from orthograph.graph_definition.models import NodeModel


# --- Domain model ---


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    released: int
    tagline: Optional[str] = None


# --- Read query ---


class MoviesByYearParams(BaseModel):
    released: int


class MoviesByYear(TypedCypherReadQueryModel[MoviesByYearParams, Movie]):
    """Return all movies released in a given year."""

    query_id = "movies_by_year"
    cypher_template = (
        "MATCH (m:Movie {released: $released}) "
        "RETURN m.title AS title, m.released AS released, m.tagline AS tagline"
    )

    def materialize(self, raw: dict[str, Any]) -> Movie:
        return Movie.model_validate(raw)


# --- Write result model ---


class CreateMovieResult(BaseModel):
    nodes_created: int


# --- Write query ---


class CreateMovieParams(BaseModel):
    title: str
    released: int


class CreateMovie(TypedCypherWriteQueryModel[CreateMovieParams, CreateMovieResult]):
    """Create a Movie node and return a structured result."""

    query_id = "create_movie"
    Output = CreateMovieResult
    cypher_template = "CREATE (m:Movie {title: $title, released: $released})"

    def interpret_result(self, raw: Any) -> CreateMovieResult:
        return CreateMovieResult(nodes_created=raw.nodes_created)


print("Domain models OK")
print(f"  MoviesByYear  Output : {MoviesByYear.Output.__name__}")
print(f"  CreateMovie   Output : {CreateMovie.Output.__name__}")

## 3. Sync path recap — caller-owned transactions

After ADR-028, `CypherExecutor.write()` runs the statement against whatever the
factory yields and **never commits**. The `_FakeSession` below does not need
`commit()` / `rollback()` — the executor no longer calls them.

This contrasts with the `06.01` fake, which had a `begin_transaction()` path
because the old executor called `session.begin_transaction()` internally.

In [ ]:
# --- In-memory store ---
_STORE: list[dict[str, Any]] = []


class _FakeCounters:
    nodes_created = 1
    nodes_deleted = 0
    relationships_created = 0
    relationships_deleted = 0
    properties_set = 0


class _FakeWriteResult:
    """Returned by _FakeSession.run() for write statements."""

    def consume(self) -> "_FakeWriteResult":
        return self

    @property
    def counters(self) -> _FakeCounters:
        return _FakeCounters()


class _FakeSession:
    """Sync fake session backed by _STORE. No transaction management (ADR-028)."""

    def __init__(self, store: list[dict[str, Any]]) -> None:
        self._store = store

    def __enter__(self) -> "_FakeSession":
        return self

    def __exit__(self, *_: Any) -> None:
        pass

    def run(self, cypher: str, **params: Any) -> Any:
        if "CREATE" in cypher:
            self._store.append(
                {
                    "title": params.get("title"),
                    "released": params.get("released"),
                    "tagline": None,
                }
            )
            return _FakeWriteResult()
        released = params.get("released")
        return [
            row
            for row in self._store
            if released is None or row["released"] == released
        ]


_sync_executor = CypherExecutor(lambda: _FakeSession(_STORE))

# Seed
_STORE.clear()
_STORE.extend(
    [
        {"title": "The Matrix", "released": 1999, "tagline": None},
        {"title": "Fight Club", "released": 1999, "tagline": None},
        {"title": "Speed", "released": 1994, "tagline": None},
    ]
)

# --- Sync read ---
movies = _sync_executor.read(MoviesByYear(), {"released": 1999})
assert len(movies) == 2
assert all(isinstance(m, Movie) for m in movies)
print("Sync read OK:", [m.title for m in movies])

# --- Sync write — executor does NOT commit ---
result = _sync_executor.write(CreateMovie(), {"title": "Inception", "released": 2010})
assert result.nodes_created == 1
print("Sync write OK: nodes_created =", result.nodes_created)
print("  executor returned result; commit was NOT called — caller owns the boundary")

## 4. Async fake session

The async driver idioms used by `AsyncCypherExecutor`:

```python
async with factory() as session:
    result = await session.run(cypher, **params)
    records = [dict(rec) async for rec in result]  # read path
    summary = (await result.consume()).counters     # write path
```

The `AsyncFakeSession` below satisfies all four of those idioms without a live
database. It is defined here as a **teaching aid only** — it is not imported from
anywhere and is not a test double.

In [ ]:
# --- Async fake internals ---


class _AsyncFakeReadResult:
    """An async-iterable result for read queries."""

    def __init__(self, rows: list[dict[str, Any]]) -> None:
        self._rows = rows

    def __aiter__(self):
        return self._async_iter()

    async def _async_iter(self):
        for row in self._rows:
            yield row


class _AsyncFakeWriteResult:
    """An async-consume result for write queries."""

    async def consume(self) -> "_AsyncFakeWriteResult":
        return self

    @property
    def counters(self) -> _FakeCounters:
        return _FakeCounters()


class AsyncFakeSession:
    """Async fake session backed by a store list.

    Satisfies the async driver idioms used by AsyncCypherExecutor:
    - async context manager       (async with factory() as session:)
    - await session.run(...)      (returns async-iterable or async-consume result)
    - async for rec in result     (read path: AsyncCypherExecutor collects records)
    - await result.consume()      (write path: AsyncCypherExecutor gets counters)

    This is a local teaching aid — it exists solely in this notebook.
    The committed / rolled_back flags let cells prove the executor never commits.
    """

    def __init__(self, store: list[dict[str, Any]]) -> None:
        self._store = store
        self.committed = False
        self.rolled_back = False

    async def __aenter__(self) -> "AsyncFakeSession":
        return self

    async def __aexit__(self, *_: Any) -> None:
        pass  # No commit — caller owns the boundary (ADR-028)

    async def run(self, cypher: str, **params: Any) -> Any:
        if "CREATE" in cypher:
            self._store.append(
                {
                    "title": params.get("title"),
                    "released": params.get("released"),
                    "tagline": None,
                }
            )
            return _AsyncFakeWriteResult()
        released = params.get("released")
        rows = [
            row
            for row in self._store
            if released is None or row["released"] == released
        ]
        return _AsyncFakeReadResult(rows)

    async def commit(self) -> None:
        self.committed = True

    async def rollback(self) -> None:
        self.rolled_back = True


print("AsyncFakeSession defined")

## 5. Async read — `AsyncCypherExecutor.read()` materialises records

`AsyncCypherExecutor` mirrors `CypherExecutor` but uses `async with`, `await
session.run()`, and `async for` to collect records. `materialize()` is called
**outside** the `async with` block — it is pure and sync.

> **IDE warning — false positive:** IDEs that do not understand Jupyter's execution
> model flag bare `await` / `async with` in notebook cells as syntax errors. They are
> not. IPython (the Jupyter kernel) has supported top-level `await` natively since
> IPython 7.0. `asyncio.run()` must **not** be used here because the kernel already
> runs an event loop; calling `asyncio.run()` inside a running loop raises
> `RuntimeError`. The `nbconvert --execute` run above confirms all cells execute
> without error.

In [ ]:
_ASYNC_STORE: list[dict[str, Any]] = [
    {"title": "The Matrix", "released": 1999, "tagline": None},
    {"title": "Fight Club", "released": 1999, "tagline": None},
    {"title": "Speed", "released": 1994, "tagline": None},
]


def _async_session_factory() -> AsyncFakeSession:
    return AsyncFakeSession(_ASYNC_STORE)


_async_executor = AsyncCypherExecutor(_async_session_factory)

# Top-level await works in Jupyter (the kernel already runs an event loop).
movies = await _async_executor.read(MoviesByYear(), {"released": 1999})
assert len(movies) == 2, f"expected 2, got {len(movies)}"
assert all(isinstance(m, Movie) for m in movies)
print("Async read OK:", [m.title for m in movies])

## 6. Async write — returns the interpreted result

`AsyncCypherExecutor.write()` awaits `result.consume()` to get mutation counters,
then calls `_summary_from_counters` and returns `query.interpret_result(summary)`.
No commit is issued.

In [ ]:
before = len(_ASYNC_STORE)
write_result = await _async_executor.write(
    CreateMovie(), {"title": "Inception", "released": 2010}
)
assert isinstance(write_result, CreateMovieResult)
assert write_result.nodes_created == 1
assert len(_ASYNC_STORE) == before + 1, "write must have run the statement"
print("Async write OK: nodes_created =", write_result.nodes_created)
print("  executor returned result; commit was NOT called — caller owns the boundary")

## 7. Caller-owned transaction — executor runs but does NOT commit

ADR-028 Decision 1: the executor **runs** the statement but the caller commits.
The `AsyncFakeSession` has a `committed` flag we can inspect directly.

The factory below yields the same session object each call so we can observe its
state after the executor returns.

In [ ]:
# Create a single session instance we can inspect after the executor call.
owned_session = AsyncFakeSession(_ASYNC_STORE)
factory = lambda: owned_session  # noqa: E731  — same object every call
executor_owned = AsyncCypherExecutor(factory)

# Run a write.
await executor_owned.write(CreateMovie(), {"title": "Interstellar", "released": 2014})

# The statement ran (node appeared in the store).
titles = [row["title"] for row in _ASYNC_STORE]
assert "Interstellar" in titles, "statement must have run"
print("Statement ran:       Interstellar is in the store")

# The executor did NOT commit.
assert owned_session.committed is False
print(
    "committed flag:     ", owned_session.committed, "← executor never calls commit()"
)

# The caller commits (simulates the transaction_context pattern in MP).
await owned_session.commit()
assert owned_session.committed is True
print("After caller commit:", owned_session.committed, "← durable only now")

## 8. Async FastAPI route — `async def` handler + `httpx.AsyncClient`

The consuming application (MP-backend) `await`s the executor inside an `async def`
FastAPI handler. `httpx.AsyncClient` verifies the HTTP interface end-to-end — no
external server required.

**Colour rule in action:** the route is `async def`; it `await`s the executor;
the executor `await`s `session.run()`. The query-definition layer (`build()`,
`materialize()`) is sync and pure — called outside the `async with` block.

In [ ]:
from fastapi import Depends, FastAPI


# Fresh store for the FastAPI demo.
_API_STORE: list[dict[str, Any]] = [
    {"title": "The Matrix", "released": 1999, "tagline": None},
    {"title": "Fight Club", "released": 1999, "tagline": None},
]

_api_executor = AsyncCypherExecutor(lambda: AsyncFakeSession(_API_STORE))


# Dependency factory — swap _api_executor here to point at a real async driver.
def get_movies_port() -> AsyncReadPort[MoviesByYearParams, Movie]:
    return AsyncQueryBackedReadPort(MoviesByYear(), _api_executor)


app = FastAPI(title="Filmography Async API")


@app.post("/movies", response_model=CreateMovieResult)
async def create_movie(body: CreateMovieParams) -> CreateMovieResult:
    """Create a Movie node. Returns mutation counters — executor does not commit."""
    return await _api_executor.write(CreateMovie(), body.model_dump())


@app.get("/movies", response_model=list[Movie])
async def get_movies(
    released: int,
    port: AsyncReadPort[MoviesByYearParams, Movie] = Depends(get_movies_port),
) -> list[Movie]:
    """Return all movies released in a given year."""
    return await port.fetch(MoviesByYearParams(released=released))


print("Registered routes:")
for route in app.routes:
    if hasattr(route, "methods"):
        print(f"  {sorted(route.methods)} {route.path}")

In [ ]:
# Test with httpx.AsyncClient — the correct async test client for ASGI apps.
transport = httpx.ASGITransport(app=app)
async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
    # --- POST: create a movie ---
    resp = await client.post("/movies", json={"title": "Inception", "released": 2010})
    assert resp.status_code == 200
    body = resp.json()
    assert body["nodes_created"] == 1
    assert "title" not in body, "write result carries counters only"
    print("POST /movies:", body)

    # --- GET: query by year ---
    resp = await client.get("/movies", params={"released": 1999})
    assert resp.status_code == 200
    movies_json = resp.json()
    assert len(movies_json) == 2
    titles_set = {m["title"] for m in movies_json}
    assert titles_set == {"The Matrix", "Fight Club"}
    print("GET /movies?released=1999:", titles_set)

    # Parse response into the declared Output model.
    parsed = [Movie.model_validate(m) for m in movies_json]
    assert all(isinstance(m, Movie) for m in parsed)
    print("Response parses into Movie objects OK")

## 9. `AsyncReadPort` / `AsyncQueryBackedReadPort` — DI pattern for async

The async port pair mirrors the sync `ReadPort` / `QueryBackedReadPort`.
`AsyncQueryBackedReadPort` holds a `(query, executor)` pair; its `fetch()` method
delegates to `await executor.read(...)`. Swap the executor at one place (the
factory) to point at a real async driver — the route never changes.

In [ ]:
port = AsyncQueryBackedReadPort(MoviesByYear(), _async_executor)
port_movies = await port.fetch(MoviesByYearParams(released=1999))
assert len(port_movies) >= 2
print(
    "AsyncQueryBackedReadPort.fetch OK:",
    [m.title for m in port_movies if m.released == 1999],
)

# The port satisfies the AsyncReadPort protocol used in Depends().
assert isinstance(port, AsyncReadPort)
print("isinstance(port, AsyncReadPort):", True)

# Sync port and async port are distinct types — colour safety.
sync_port = QueryBackedReadPort(MoviesByYear(), _sync_executor)
assert not isinstance(sync_port, AsyncReadPort)
print("sync port is NOT AsyncReadPort  — colour safety enforced")

## Summary

| | Sync path | Async path |
|---|---|---|
| Executor | `CypherExecutor` | `AsyncCypherExecutor` |
| Port | `QueryBackedReadPort` | `AsyncQueryBackedReadPort` |
| Session idiom | `with factory() as s:` | `async with factory() as s:` |
| Run statement | `s.run(cypher, **p)` | `await s.run(cypher, **p)` |
| Iterate records | `list(result)` | `[r async for r in result]` |
| Consume counters | `result.consume().counters` | `(await result.consume()).counters` |
| Commits? | **No** (ADR-028) | **No** (ADR-028) |
| Query-definition layer | colour-neutral, shared | colour-neutral, shared |

The query-definition layer (`build()` / `materialize()` / `interpret_result()`) is
**colour-neutral**: sync, pure, called outside the `async with` block. Neither
executor duplicates any query logic.